In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from collections import Counter
import re
import time

from datasets import load_dataset

dataset = load_dataset("imdb")
train_data = dataset["train"]
test_data = dataset["test"]

print(f"Train dataset: {len(train_data)}")
print(f"Test dataset: {len(test_data)}")

print(f"\nlabel: {train_data[0]['label']}")
print(f"First 200 words of comment: {train_data[0]['text'][:200]}")


Train dataset: 25000
Test dataset: 25000

label: 0
First 200 words of comment: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev


In [2]:
def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()

def build_vocab(texts, max_vocab=20000):
    counter = Counter()
    for text in texts:
        counter.update(simple_tokenize(text))

    vocab = {'<pad>': 0, '<unk>': 1}
    for word, count in counter.most_common(max_vocab):
        vocab[word] = len(vocab)

    return vocab

vocab = build_vocab(train_data['text'])
print(f"vocabulary size: {len(vocab)}")

vocabulary size: 20002


In [3]:
def encode_text(text, vocab, max_len=256):
    tokens = simple_tokenize(text)[:max_len]
    ids = [vocab.get(t, vocab['<unk>']) for t in tokens]
    return ids

In [4]:
def collate_fn(batch):
    texts = [item['text'] for item in batch]
    labels = [item['label'] for item in batch]

    encoded = [encode_text(t, vocab) for t in texts]
    max_len = max(len(seq) for seq in encoded)

    padded = []
    lengths = []
    for seq in encoded:
        lengths.append(len(seq))
        padded.append(seq + [0]*(max_len - len(seq)))
    return (
        torch.tensor(padded, dtype = torch.long),
        torch.tensor(lengths, dtype = torch.long),
        torch.tensor(labels, dtype=torch.float32)
    )

BATCH_SIZE = 64
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

texts, lengths, labels = next(iter(train_loader))
print(f"texts shape: {texts.shape}")
print(f"lengths shape: {lengths.shape}")
print(f"labels shape: {labels.shape}")
print(f"lengths range: {lengths.min()} ~ {lengths.max()}")

texts shape: torch.Size([64, 256])
lengths shape: torch.Size([64])
labels shape: torch.Size([64])
lengths range: 36 ~ 256


In [5]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size=128, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_size, hidden_size, 
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        # bidirectional所以hidden_size * 2
        self.fc = nn.Linear(hidden_size*2, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, lengths):
        embed = self.embedding(x)
        embed = self.dropout(embed)

        from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
        packed = pack_padded_sequence(embed, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, (h_n, c_n) = self.lstm(packed)

        # h_n shape: (num_layers * directions, batch, hidden_size)
        # h_n[0] = 第1层 forward 的最终hidden
        # h_n[1] = 第1层 backward 的最终hidden
        # h_n[2] = 第2层 forward 的最终hidden
        # h_n[3] = 第2层 backward 的最终hidden
        hidden = torch.cat([h_n[-2], h_n[-1]], dim=1) # (batch, 2*hidden_size)

        hidden = self.dropout(hidden)
        logit = self.fc(hidden).squeeze(1) # (batch, )
        return logit
        
model = SentimentLSTM(len(vocab))
print(model)
print(f"参数量: {sum(p.numel() for p in model.parameters()):,}")


SentimentLSTM(
  (embedding): Embedding(20002, 128, padding_idx=0)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)
参数量: 3,219,969


In [6]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(device)

model = SentimentLSTM(len(vocab)).to(device)

mps


In [7]:
# 在训练之前检查数据
batch_texts, batch_lengths, batch_labels = next(iter(train_loader))
print(f"labels分布: {batch_labels.sum().item()}/{batch_labels.shape[0]} 是正面")
print(f"lengths前10个: {batch_lengths[:10]}")
print(f"texts第一条前20个token: {batch_texts[0][:20]}")
print(f"texts第一条里0的个数: {(batch_texts[0]==0).sum().item()}")

# 检查vocab
print(f"\nvocab大小: {len(vocab)}")
test_text = "this movie was great"
print(f"'{test_text}' → {encode_text(test_text, vocab)}")
test_text2 = "terrible awful waste"
print(f"'{test_text2}' → {encode_text(test_text2, vocab)}")

labels分布: 33.0/64 是正面
lengths前10个: tensor([137, 256, 256,  97, 147, 256, 256, 135, 256, 242])
texts第一条前20个token: tensor([  11,   14,    4,  179,  527,   18,   11,   18,    7,   50,    6,   41,
         849,  187,    3,  104,    3,   28, 2197,   41])
texts第一条里0的个数: 119

vocab大小: 20002
'this movie was great' → [11, 18, 14, 85]
'terrible awful waste' → [384, 381, 424]


In [8]:
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
criterion = nn.BCEWithLogitsLoss()

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for texts, lengths, labels in loader:
        texts = texts.to(device)
        labels = labels.to(device)

        logits = model(texts, lengths)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += texts.size(0) * loss.item()
        preds = (logits > 0).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, 100.0 * correct /total
    
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for texts, lengths, labels in loader:
            texts = texts.to(device)
            labels = labels.to(device)

            logits = model(texts, lengths)
            loss = criterion(logits, labels)
            
            total_loss += texts.size(0) * loss.item()
            preds = (logits > 0).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    return total_loss / total, 100.0 * correct / total, all_preds, all_labels


NUM_EPOCHS = 8

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | "
      f"{'Test Loss':>9} | {'Test Acc':>10} | {'LR':>10}")
print("-"*75)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False, collate_fn=collate_fn)

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion, device)

    scheduler.step(test_loss)
    current_lr = optimizer.param_groups[0]['lr']

    elapsed = time.time() - start
    print(f"{epoch:>5} | {train_loss:>10.4f} | {train_acc:>8.2f}% | "
          f"{test_loss:>9.4f} | {test_acc:>7.2f}% | {current_lr:>10.6f}")
    



Epoch | Train Loss | Train Acc | Test Loss |   Test Acc |         LR
---------------------------------------------------------------------------
    1 |     0.6245 |    64.66% |    0.5469 |   72.05% |   0.001000
    2 |     0.4962 |    76.05% |    0.4583 |   79.86% |   0.001000
    3 |     0.3988 |    82.16% |    0.3856 |   83.26% |   0.001000
    4 |     0.3272 |    86.14% |    0.3832 |   84.36% |   0.001000
    5 |     0.2875 |    88.15% |    0.3243 |   86.42% |   0.001000
    6 |     0.2412 |    90.38% |    0.3223 |   86.97% |   0.001000
    7 |     0.2184 |    91.41% |    0.3279 |   86.65% |   0.001000
    8 |     0.1980 |    92.23% |    0.3370 |   87.29% |   0.001000


***Change to use entire test dataset***

In [9]:
test_loader = DataLoader(test_data, batch_size=256, shuffle=False, collate_fn=collate_fn)

_, test_acc, all_preds, all_labels = evaluate(model, test_loader, criterion, device)

preds_t = torch.tensor(all_preds)
labels_t = torch.tensor(all_labels)

TP = ((preds_t == 1) & (labels_t == 1)).sum().item()
FP = ((preds_t == 1) & (labels_t == 0)).sum().item()
TN = ((preds_t == 0) & (labels_t == 0)).sum().item()
FN = ((preds_t == 0) & (labels_t == 1)).sum().item()

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2*precision*recall/(precision+recall) if (precision + recall) > 0 else 0

print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Precision:     {precision:.4f}")
print(f"Recall:        {recall:.4f}")
print(f"F1 score:      {f1:.4f}")
print(f"\nConfusion Matrix:")
print(f"               Pred Neg  Pred Pos")
print(f"   True Neg    {TN:>7}   {FP:>7}")
print(f"   True Pos    {FN:>7}   {TP:>7}")

Test Accuracy: 87.29%
Precision:     0.8437
Recall:        0.9153
F1 score:      0.8781

Confusion Matrix:
               Pred Neg  Pred Pos
   True Neg      10381      2119
   True Pos       1059     11441


In [11]:
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab': vocab,
    'test_accuracy': test_acc, 
    'f1_score': f1,
    'architecture': 'BiLSTM 2-layer, embed=128, hidden=128, dropout=0.3'
}, "imdb_sentiment_checkpoint.pth")

def predict_sentiment(text, model, vocab, device):
    model.eval()
    ids = encode_text(text, vocab)
    x = torch.tensor([ids], dtype=torch.long).to(device)
    length = torch.tensor([len(ids)])
    with torch.no_grad():
        logit = model(x, length)
        prob = torch.sigmoid(logit).item()
    sentiment = "Positive" if prob > 0.5 else "Negative"
    return sentiment, prob


# 测试几个例子
tests = [
    "This movie was absolutely wonderful and I loved every minute of it",
    "Terrible film, waste of time, the acting was horrible",
    "It was okay, nothing special but not bad either",
]
for t in tests:
    sent, prob = predict_sentiment(t, model, vocab, device)
    print(f"{sent} ({prob:.2f}): {t[:60]}...")
    

Positive (1.00): This movie was absolutely wonderful and I loved every minute...
Negative (0.00): Terrible film, waste of time, the acting was horrible...
Negative (0.00): It was okay, nothing special but not bad either...
